In [1]:
import pandas as pd

In [2]:
df=pd.read_csv('../data/ann_customer_churn_50000.csv')

In [3]:
df.head()

,age,tenure_months,monthly_charges,support_tickets,late_payments,usage_hours,satisfaction,contract_monthly,tech_support,auto_payment,churn
0,23,7,76.20,5,1,27.58,9.87,1,1,0,0
1,62,69,53.67,5,1,28.71,6.41,1,0,1,0
2,55,38,86.48,4,1,39.27,4.58,0,1,1,0
3,43,72,79.26,0,1,55.58,7.23,0,1,0,0
4,43,30,69.54,6,1,35.25,5.19,1,0,1,0


In [4]:
df.shape

(50000, 11)

In [5]:
df.isna().sum()

age                 0
tenure_months       0
monthly_charges     0
support_tickets     0
late_payments       0
usage_hours         0
satisfaction        0
contract_monthly    0
tech_support        0
auto_payment        0
churn               0
dtype: int64

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   age               50000 non-null  int64  
 1   tenure_months     50000 non-null  int64  
 2   monthly_charges   50000 non-null  float64
 3   support_tickets   50000 non-null  int64  
 4   late_payments     50000 non-null  int64  
 5   usage_hours       50000 non-null  float64
 6   satisfaction      50000 non-null  float64
 7   contract_monthly  50000 non-null  int64  
 8   tech_support      50000 non-null  int64  
 9   auto_payment      50000 non-null  int64  
 10  churn             50000 non-null  int64  
dtypes: float64(3), int64(8)
memory usage: 4.2 MB


<h2>Feature Selection</h2>

In [7]:
X=df.drop(columns=['churn'],axis=1)
y=df['churn']

In [8]:
from sklearn.model_selection import train_test_split

In [9]:
X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [10]:
from sklearn.preprocessing import StandardScaler

In [11]:
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)

In [12]:
X_test_scaled=scaler.transform(X_test)

In [13]:
y_train.to_numpy()

array([0, 0, 0, ..., 0, 0, 0], shape=(40000,))

In [14]:
y_test.to_numpy()

array([0, 0, 1, ..., 0, 0, 0], shape=(10000,))

<h3>Preparing data for pytorchby converting them to tensors</h3>

In [15]:
import torch

In [16]:
X_train_scaled_tensor=torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy()).reshape(-1, 1)

X_test_scaled_tensor=torch.tensor(X_test_scaled,dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.to_numpy()).reshape(-1, 1)

In [17]:
y_train_tensor = y_train_tensor.float()
y_test_tensor = y_test_tensor.float()

In [18]:
print(X_train_scaled_tensor.shape)
print(y_train_tensor.shape)

print(X_test_scaled_tensor.shape)
print(y_test_tensor.shape)

torch.Size([40000, 10])
torch.Size([40000, 1])
torch.Size([10000, 10])
torch.Size([10000, 1])


<h2>Neural Network Model 🧠</h2>

In [19]:
import torch.nn as nn

In [20]:
neural_model=nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
    
)

In [21]:
neural_model

Sequential(
  (0): Linear(in_features=10, out_features=32, bias=True)
  (1): ReLU()
  (2): Linear(in_features=32, out_features=16, bias=True)
  (3): ReLU()
  (4): Linear(in_features=16, out_features=8, bias=True)
  (5): ReLU()
  (6): Linear(in_features=8, out_features=1, bias=True)
)

In [22]:
pos_weight = (y_train_tensor == 0).sum() / (y_train_tensor == 1).sum()

print(pos_weight)

tensor(12.8026)


In [23]:
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [24]:
optimizer = torch.optim.Adam(
    neural_model.parameters(),
    lr=0.001
)

In [25]:
for epoch in range (1,501):
    optimizer.zero_grad()
    y_pred=neural_model(X_train_scaled_tensor)
    loss=loss_fn(y_pred,y_train_tensor)
    loss.backward()
    optimizer.step()

    print(f'{epoch}: {loss}')

1: 1.2918822765350342
2: 1.2908565998077393
3: 1.2898404598236084
4: 1.2888330221176147
5: 1.2878249883651733
6: 1.2868142127990723
7: 1.2857998609542847
8: 1.284772515296936
9: 1.2837251424789429
10: 1.2826496362686157
11: 1.2815468311309814
12: 1.2804162502288818
13: 1.279256820678711
14: 1.278059959411621
15: 1.2768226861953735
16: 1.2755472660064697
17: 1.2742353677749634
18: 1.2728825807571411
19: 1.2714892625808716
20: 1.2700514793395996
21: 1.2685661315917969
22: 1.267027497291565
23: 1.2654314041137695
24: 1.2637814283370972
25: 1.2620741128921509
26: 1.2603031396865845
27: 1.2584657669067383
28: 1.2565550804138184
29: 1.2545697689056396
30: 1.252503752708435
31: 1.2503535747528076
32: 1.2481173276901245
33: 1.2457900047302246
34: 1.2433661222457886
35: 1.2408417463302612
36: 1.23820960521698
37: 1.2354644536972046
38: 1.2325994968414307
39: 1.2296091318130493
40: 1.2264841794967651
41: 1.2232106924057007
42: 1.2197835445404053
43: 1.2162007093429565
44: 1.2124520540237427
45: 

In [26]:
y_test_pred = neural_model(X_test_scaled_tensor)

In [27]:
y_test_pred.shape

torch.Size([10000, 1])

In [28]:
loss_fn(y_test_pred,y_test_tensor)

tensor(0.8658, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)

In [29]:
y_test_pred_classes=(y_test_pred>=0.5).float()

In [30]:
y_test_pred_classes[:10]

tensor([[0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.]])

In [31]:
torch.unique(y_test_pred_classes,return_counts=True)

(tensor([0., 1.]), tensor([8248, 1752]))

In [32]:
print(y_test_pred.min())
print(y_test_pred.max())

tensor(-7.7149, grad_fn=<MinBackward1>)
tensor(5.2410, grad_fn=<MaxBackward1>)


In [33]:
print(torch.unique(y_test_tensor, return_counts=True))

(tensor([0., 1.]), tensor([9275,  725]))


In [34]:
print(y_test_pred[y_test_tensor == 1].mean())
print(y_test_pred[y_test_tensor == 0].mean())

tensor(1.2303, grad_fn=<MeanBackward0>)
tensor(-1.4395, grad_fn=<MeanBackward0>)


In [35]:
from sklearn.metrics import classification_report
print(classification_report(y_test_tensor.numpy(), y_test_pred_classes.numpy()))

              precision    recall  f1-score   support

         0.0       0.97      0.86      0.91      9275
         1.0       0.28      0.68      0.40       725

    accuracy                           0.85     10000
   macro avg       0.63      0.77      0.66     10000
weighted avg       0.92      0.85      0.88     10000



In [36]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test_tensor.numpy(),
    y_test_pred_classes.numpy()
)

print(cm)

[[8016 1259]
 [ 232  493]]


In [37]:
print(y_test_pred.min())
print(y_test_pred.max())
print(y_test_pred.mean())

tensor(-7.7149, grad_fn=<MinBackward1>)
tensor(5.2410, grad_fn=<MaxBackward1>)
tensor(-1.2459, grad_fn=<MeanBackward0>)


In [38]:
y_test_pred_class_03 = (y_test_pred >= 0.3).float()

print(torch.unique(y_test_pred_class_03, return_counts=True))

(tensor([0., 1.]), tensor([7947, 2053]))


In [39]:
cm_03 = confusion_matrix(
    y_test_tensor.numpy(),
    y_test_pred_class_03.numpy()
)

print(cm_03)

[[7743 1532]
 [ 204  521]]


In [40]:
print("Actual churners:", y_test_pred[y_test_tensor == 1].mean())
print("Actual non-churners:", y_test_pred[y_test_tensor == 0].mean())

Actual churners: tensor(1.2303, grad_fn=<MeanBackward0>)
Actual non-churners: tensor(-1.4395, grad_fn=<MeanBackward0>)


In [41]:
test_loss = loss_fn(y_test_pred, y_test_tensor)

print("Train loss:", loss.item())
print("Test loss:", test_loss.item())

Train loss: 0.7846903204917908
Test loss: 0.8658000230789185


In [42]:
y_test_pred = neural_model(X_test_scaled_tensor)

y_test_pred_class = (y_test_pred >= 0.5).float()

cm = confusion_matrix(
    y_test_tensor.numpy(),
    y_test_pred_class.numpy()
)

print(cm)

[[8016 1259]
 [ 232  493]]


In [43]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    y_test_tensor.numpy(),
    y_test_pred_class.numpy()
)

print(accuracy)

0.8509


In [44]:
y_test_pred = neural_model(X_test_scaled_tensor)
y_test_pred_class = (y_test_pred >= 0.5).float()

cm = confusion_matrix(
    y_test_tensor.numpy(),
    y_test_pred_class.numpy()
)

print(cm)

[[8016 1259]
 [ 232  493]]


In [45]:
y_test_logits = neural_model(X_test_scaled_tensor)

y_test_pred = torch.sigmoid(y_test_logits)

y_test_pred_class = (y_test_pred >= 0.5).float()

In [46]:
cm = confusion_matrix(
    y_test_tensor.numpy(),
    y_test_pred_class.numpy()
)

print(cm)

[[7326 1949]
 [ 164  561]]


In [47]:
import numpy as np

In [48]:
thresholds=np.arange(0.3,0.8,0.1)

In [54]:
for threshold in thresholds:
    y_predct_th=(y_test_pred>=threshold).float()

    cm_t=confusion_matrix(
     y_test_tensor.numpy(),
     y_predct_th.numpy()
    
    )
    print(f'Threshold {threshold} -> {cm_t}')

Threshold 0.3 -> [[5824 3451]
 [  92  633]]
Threshold 0.4 -> [[6634 2641]
 [ 116  609]]
Threshold 0.5 -> [[7326 1949]
 [ 164  561]]
Threshold 0.6000000000000001 -> [[7875 1400]
 [ 224  501]]
Threshold 0.7000000000000002 -> [[8378  897]
 [ 271  454]]


In [55]:
y_test_logits = neural_model(X_test_scaled_tensor)
y_test_pred = torch.sigmoid(y_test_logits)

y_pred_final = (y_test_pred >= 0.3).float()

In [60]:
import torch
import joblib

torch.save(neural_model.state_dict(), "../models/churn_model.pth")

joblib.dump(scaler, "../models/scaler.pkl")

['../models/scaler.pkl']